# 03 — Models
Predict GPA from study habits and AI behavior.

⚠️ With the current sample size, results are indicative only. XGBoost is included but will be most meaningful once n reaches 200+.

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../data/clean.csv')
FEATURES = ['study_hours_n', 'sleep_n', 'consistency', 'ai_freq', 'ai_confidence',
            'ai_dependence', 'ai_verify', 'difficulty', 'distraction', 'independence']
data = df[FEATURES + ['gpa']].dropna()
X, y = data[FEATURES], data['gpa']
print(f"n = {len(data)}")

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear regression': LinearRegression(),
    'Random forest': RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42),
}
for name, m in models.items():
    r2 = cross_val_score(m, X, y, cv=cv, scoring='r2')
    mae = -cross_val_score(m, X, y, cv=cv, scoring='neg_mean_absolute_error')
    print(f"{name:20s}  R2 = {r2.mean():.3f}  MAE = {mae.mean():.3f}")

In [ ]:
# Which features drive the random forest?
rf = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42).fit(X, y)
pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

In [ ]:
# Linear regression coefficients (direction of effect)
lr = LinearRegression().fit(X, y)
pd.Series(lr.coef_, index=FEATURES).sort_values(key=abs, ascending=False).round(3)

### XGBoost (enable when n ≥ 200)
```python
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05)
print(cross_val_score(xgb, X, y, cv=cv, scoring='r2').mean())
```